# SQL query from table names - Continued

In [5]:
import os
from dotenv import load_dotenv


load_dotenv() # loads .env into the environment

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set")

## The old Prompt

In [6]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "name": "ID_usr",
      "tipo": "int"
    },
    {
      "name": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [7]:
context = [ {'role':'system', 'content':"""
 CREATE TABLE employees (
    employee_id INT PRIMARY KEY,
    name VARCHAR(255),
    hire_date DATE
);
             
-- Example rows:
-- (1, 'Alice Novak', '2020-03-15')
-- (2, 'Lukas Meyer', '2018-07-01')
-- (3, 'Sofia Rossi', '2022-01-10')

CREATE TABLE salary (
    employee_id INT,
    year INT,
    amount DECIMAL(10,2),
    PRIMARY KEY (employee_id, year),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);
-- Example rows:
-- (1, 2022, 85000.00)
-- (1, 2023, 90000.00)
-- (2, 2023, 120000.00)

CREATE TABLE studies (
    employee_id INT,
    degree_level VARCHAR(50),
    PRIMARY KEY (employee_id, degree_level),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id)
);

-- Example rows:
-- (1, 'Bachelor')
-- (2, 'PhD')
-- (3, 'Master')

CREATE TABLE departments (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(100)
);

-- Example rows:
-- (10, 'Engineering')
-- (20, 'Finance')
-- (30, 'Human Resources')

CREATE TABLE employee_department (
    employee_id INT,
    department_id INT,
    start_date DATE,
    PRIMARY KEY (employee_id, department_id),
    FOREIGN KEY (employee_id) REFERENCES employees(employee_id),
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

-- Example rows:
-- (1, 10, '2020-03-15')
-- (2, 20, '2018-07-01')
-- (3, 10, '2022-01-10')
             
"""} ]



In [8]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
Question: When did Alice Novak join the company?
SELECT hire_date
FROM employees
WHERE name = 'Alice Novak';;
 Question: What was the average salary in 2023?
SELECT AVG(amount) AS average_salary
FROM salary
WHERE year = 2023;
Question: Which departments had employees hired in 2023?
SELECT DISTINCT d.department_name
FROM employees e
JOIN employee_department ed
  ON e.employee_id = ed.employee_id
JOIN departments d
  ON ed.department_id = d.department_id
WHERE e.hire_date >= '2023-01-01'
  AND e.hire_date < '2024-01-01';
"""
})

In [9]:
from openai import OpenAI

#Function to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI()

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [10]:
#new
context_user = context.copy()
print(return_CCRMSQL("What is the average salary in 2023 for users with a PhD?", context_user))

SELECT AVG(s.amount) AS average_salary
FROM salary s
JOIN employees e ON s.employee_id = e.employee_id
JOIN studies st ON e.employee_id = st.employee_id
WHERE s.year = 2023
AND st.degree_level = 'PhD';


In [11]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What is the average salary in 2023 for users with a PhD?", old_context_user))

This is your SQL:
```sql
SELECT AVG(s.salary) AS average_salary
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE st.educational_level = 'PhD' AND YEAR(s.year) = 2023;
```

This SQL query retrieves the average salary for users with a PhD in the year 2023. It joins the tables `employees`, `studies`, and `salary` on the user ID, filters the results to only include users with a PhD, and limits the results to the year 2023. Finally, it calculates the average salary for these users.


In [12]:
#new
context_user = context.copy()
print(return_CCRMSQL("List users who have salary records but no studies records", context_user))

SELECT e.name
FROM employees e
LEFT JOIN salary s ON e.employee_id = s.employee_id
LEFT JOIN studies st ON e.employee_id = st.employee_id
WHERE st.employee_id IS NULL
AND s.employee_id IS NOT NULL;


In [13]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("List users who have salary records but no studies records", old_context_user))

This is your SQL:
```sql
SELECT e.ID_usr, e.name
FROM employees e
LEFT JOIN salary s ON e.ID_usr = s.ID_usr
LEFT JOIN studies st ON e.ID_usr = st.ID_usr
WHERE st.ID IS NULL;
```

This SQL query selects the users from the employees table who have salary records in the salary table but do not have any corresponding records in the studies table. It uses a LEFT JOIN to combine the tables and filters out the records where the studies ID is NULL, indicating that there are no studies records for that user.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [14]:
#new
context_user = context.copy()
print(return_CCRMSQL("What is the average salary in 2023 by department?", context_user))

SELECT d.department_name, AVG(s.amount) AS average_salary
FROM salary s
JOIN employees e ON s.employee_id = e.employee_id
JOIN employee_department ed ON e.employee_id = ed.employee_id
JOIN departments d ON ed.department_id = d.department_id
WHERE s.year = 2023
GROUP BY d.department_name;


In [ ]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("What is the average salary in 2023 by department?", old_context_user))

#Output logic is not correct. Employees as no departmente column -> Schema hallucination

This is your SQL:
```sql
SELECT e.department, AVG(s.salary) AS avg_salary
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.year = '2023'
GROUP BY e.department;
```

This SQL query retrieves the average salary in 2023 by department by joining the "employees" and "salary" tables on the employee ID, filtering the data for the year 2023, and then calculating the average salary for each department.


In [17]:
#new
context_user = context.copy()
print(return_CCRMSQL("How many employees in the Engineering department have a Master's degree and earned more than 100000 in 2023?", context_user))

SELECT COUNT(*) AS employee_count
FROM employees e
JOIN employee_department ed
  ON e.employee_id = ed.employee_id
JOIN departments d
  ON ed.department_id = d.department_id
JOIN salary s
  ON e.employee_id = s.employee_id
JOIN studies st
  ON e.employee_id = st.employee_id
WHERE d.department_name = 'Engineering'
  AND st.degree_level = 'Master'
  AND s.year = 2023
  AND s.amount > 100000;


In [18]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("How many employees in the Engineering department have a Master's degree and earned more than 100000 in 2023?", old_context_user))

#Output logic is not correct. Employees as no departmente column -> Schema hallucination

This is your SQL:
```sql
SELECT COUNT(*) 
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
JOIN salary sa ON e.ID_usr = sa.ID_usr
WHERE s.educational_level = 'Master' 
AND s.Speciality = 'Engineering' 
AND sa.salary > 100000 
AND YEAR(sa.year) = 2023;
```

This SQL query retrieves the count of employees who are in the Engineering department, have a Master's degree, and earned more than 100000 in the year 2023 by joining the employees, studies, and salary tables and applying the specified conditions.


## Summary

Old system:
→ Pattern completion
→ Semantic guesswork
→ Hallucinates missing structure

New system:
→ Constraint-driven reasoning
→ Follows actual relational graph
→ Maintains schema fidelity

SQL DDL grounding shifts model behavior from semantic approximation to relational reasoning, significantly reducing schema hallucination under multi-hop conditional queries.